In [1]:
import openai
import config

C:\Users\moham\anaconda3\envs\llms_course_env\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
api_key = config.api_key

In [3]:
# to initialize openai client 
openai.api_key = api_key

In [4]:
from langchain.document_loaders import WebBaseLoader 
from langchain.text_splitter import RecursiveCharacterTextSplitter 
from langchain.embeddings import OpenAIEmbeddings 
from langchain.vectorstores import FAISS 
from langchain.memory import ConversationBufferMemory 
from langchain.llms import OpenAI 
from langchain.chains import ConversationalRetrievalChain 
from langchain.chat_models import ChatOpenAI

### our use case to read the latest info from 365 Ds WEBSITE
##### so we use a webbaseloader to read from url

In [25]:
url = "https://365datascience.com/courses/"

In [26]:
loader = WebBaseLoader(url)

In [27]:
## CREATE A LANG CHAIN DOCUMENT OBJECT
raw_documents = loader.load()

In [28]:
## NOW WE SHOULD TAKE OUR DATA AND CHUNNK IT UP INTO SMALL PIECES
#This is necessary in order to make sure we only pass the smallest, most relevant pieces of text to
#the language model.

#So we'll create our text splitter.

#And then use the recursive character text splitter.

text_splitter = RecursiveCharacterTextSplitter()
documents = text_splitter.split_documents(raw_documents)

#this will take thetext from the url and break it 

In [29]:
#Once we have these smaller chunks, we can create our embeddings for each piece of text and store them

#in a vector store to create our embeddings We'll use OpenAI embeddings and then we need to specify our API key
embeddings = OpenAIEmbeddings(openai_api_key = api_key)

In [30]:
#We can then create our vector store.
vectorstore  = FAISS.from_documents(documents,embeddings)

#### We can now create a memory object which is necessary to track the inputs and outputs, and for the model

#### to hold a conversation.

In [31]:
#So to create our memory we use conversation buffer memory.
memory = ConversationBufferMemory(memory_key = "chat_history", return_messages = True)

### We can now initialize the conversational retrieval chain.

#### This chain first combines the chat history and the question asked to the model into a standalone question.

#### It then looks up the relevant documents from the retriever and finally passes those documents and the

#### question to a question answering chain to return the correct response.

In [36]:
# So let's set up our chain.
qa = ConversationalRetrievalChain.from_llm(ChatOpenAI(openai_api_key = api_key,
                                                      model = "gpt-3.5-turbo",
                                                      temperature = 0),
                                                      vectorstore.as_retriever(),
                                                      memory= memory)


### We're now ready to ask our model questions based on this text we provided from the 365 Data Science website.

In [37]:
#So we'll use our QA chain and we'll pass our query to our chain.
query = "what is the next course to be uploaded on 365DataScience platform?"

In [38]:
result = qa({"question": query})

In [39]:
#To get our results we just need to access the answer from the results.

result["answer"]

'To find out about the next course to be uploaded on the 365DataScience platform, you can regularly check their website or subscribe to their newsletter. This way, you will receive updates and notifications about new courses as soon as they are available.'